In [50]:
import docx
import pandas as pd
import re
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pathlib
import json
import shutil

## Read Word Doc

In [2]:
doc = docx.Document("test.docx")

In [6]:

for index,p in enumerate(doc.paragraphs):
    a=p.text
    a=a.strip()
    if a.find("File:") > -1:
       b=a.split()
       file=b[-1]
       file=file.strip()
       print(file)

char_orgs_ext.txt
char_orgs_sol.txt
doing_bus_as.txt
offices.txt
persons_entity.txt
persons_sols_ntcs.txt
purpose.txt
reg_finan.txt
(cont)
(continued)
sn_cmpgn_rpts.txt
sol_ntcs.txt
(continued)
sol_ntcs_locs.txt
sol_typ_entity.txt
sol_typ_sol_ntcs.txt
states.txt
tax_exempt_cds.txt


## Extract and write out all the tables

In [ ]:
ntable=0
for table in doc.tables[1:]:
    ntable+=1
    fout=open(f"defs/table-{ntable}.tsv","w")
    fout.write("Field\tDescription\n")
    for index,row in enumerate(table.rows):
        for cell in row.cells:
            line=cell.text
            line=line.strip()
            if line.find("Description") == -1:
                a=re.findall('\(.*?\)',line)
                if len(a) > 0: 
                    ed = line.rfind("(")
                    var = re.sub("[(|)]","",a[-1])
                    print(ntable,var,"\t",line[:ed])
                    fout.write(f"{var}\t{line[:ed]}\n")
                else:
                    var=""
                    print(ntable,var,"\t",line)
                    fout.write(f"{var}\t{line}\n")
                    
    fout.close()
    print("------")

##  Get File Names, Descriptions from First Table

In [24]:
cdos = {}
for index,row in enumerate(doc.tables[0].rows):
    file= row.cells[0].text.strip()
    descp = row.cells[1].text.strip()
    cdos[file]=descp
    # for cell in row.cells:
    #     line=cell.text
    #     line=line.strip()
    #     print(line)

In [42]:
cdos

{'File Name': 'Description',
 'char_orgs_ext.txt': 'Charitable Organization requests for extensions for financial reporting',
 'char_orgs_sol.txt': 'Charitable Organization’s Paid Solicitors',
 'doing_bus_as.txt': '‘Doing business as’ names associated with Charitable Organization',
 'offices.txt': 'Offices associated with Charitable Organizations, Paid Solicitors or Professional Fundraising Consultants',
 'persons_entity.txt': 'People associated with Charitable Organizations, Paid Solicitors or Professional Fundraising Consultants',
 'persons_sols_ntcs.txt': 'People associated with solicitation notices',
 'purpose.txt': 'Purpose of the Charitable Organization',
 'reg_finan.txt': 'Registration and financial data for Charitable Organizations, Paid Solicitors and Professional Fundraising Consultants',
 'sn_cmpgn_rpts.txt': 'Campaign reports for solicitation notices',
 'sol_ntcs.txt': 'Solicitation notice information',
 'sol_ntcs_locs.txt': 'Solicitation notice locations',
 'sol_typ_entity

## Read Cron Dataset Titles and Files

In [85]:
df = pd.read_csv("cron-files-titles.tsv",delimiter=":")

In [86]:
df.head()

,file,Title
0,tax_exempt_cds.txt,Federal Tax-Exempt Subsection Codes in C...
1,reg_finan.txt,"Registration for Charities, Paid Solicit..."
2,offices.txt,Charitable Organizations’ Offices in Col...
3,states.txt,Other State Solicitation of Charities’ R...
4,purpose.txt,Charitable Purpose of the Charity in Col...


In [87]:
df.columns
df.columns = ['File','Title']

## Xreference Cron Tiles and File Names from Word Document

In [88]:
def setTitle(row):
    file = row['File'].strip()
    title = row['Title'].strip()
    desc=""
    if file in cdos:
        desc  = cdos[file]
    file = file.replace(".txt",".tsv")
    row['File'] = file
    row['Description'] = desc
    row['Title'] = title
    return row
    
df['Description'] = ""

df = df.apply(setTitle,axis=1)

In [89]:
df.columns

Index(['File', 'Title', 'Description'], dtype='object')

In [90]:
df.head(20)

,File,Title,Description
0,tax_exempt_cds.tsv,Federal Tax-Exempt Subsection Codes in Colorado,Tax exempt codes and descriptions
1,reg_finan.tsv,"Registration for Charities, Paid Solicitors, P...",Registration and financial data for Charitable...
2,offices.tsv,Charitable Organizations’ Offices in Colorado,Offices associated with Charitable Organizatio...
3,states.tsv,Other State Solicitation of Charities’ Registr...,States associated with Charitable Organization...
4,purpose.tsv,Charitable Purpose of the Charity in Colorado,Purpose of the Charitable Organization
5,sol_ntcs.tsv,Paid Solicitor Solicitation Notices in Colorado,Solicitation notice information
6,sn_cmpgn_rpts.tsv,Campaign Reports for Solicitation Notices to C...,Campaign reports for solicitation notices
7,persons_sol_ntcs.tsv,Solicitation Campaign Supervisors Listed on So...,
8,char_orgs_ext.tsv,Charity Extension Requests,Charitable Organization requests for extension...
9,persons_entity.tsv,Persons Associated with Charitable Organizatio...,People associated with Charitable Organization...


In [91]:
df.to_csv("defs/xrefs.tsv",sep="\t",index=False)

## Add 4x4 to Dataset Dictionaries, Convert to JSON

In [69]:
def init():  
    
##  get cdos nonprofit charity definition file name to list
    info={}
    desktop = pathlib.Path("/home/joe/bic_etl")
    runEtls = []
    desktop.rglob("*")
    files = list(desktop.rglob("*"))
# Which you can wrap in a list() constructor to materialize
    for ff in files:
        if (str(ff).split("/")[-1] == "run_etl.json"):     
            runEtls.append(ff)

 #   print(f"{len(runEtls)} run_etl.json files found")    
    bicHome="/home/joe/bic_etl"
    start=len(bicHome)
##  Process the run_etl.json files
    for file in runEtls:
      f = open(file,"r")
      data = json.load(f)
      groupNew = str(file)[start:-12]
      
    #  groups.append(ll)
      for val in data:

 #   for val in dataSets:
        if "title" in val:
          title=val["title"].strip()
          info[title] = groupNew
        
    return info
   
            

info=init()              

In [70]:
info

{'CIM Catalog Download': '/catalog/',
 'Durable Medical Equipment Suppliers in Colorado': '/cdos/health/',
 'Current Notaries in Colorado': '/cdos/government/',
 'Uniform Commercial Code (UCC) Collateral Information in Colorado': '/cdos/business/ucc/',
 'Uniform Commercial Code (UCC) Debtor Information in Colorado': '/cdos/business/ucc/',
 'Uniform Commercial Code (UCC) Filing Information in Colorado': '/cdos/business/ucc/',
 'Secured Party Information in Colorado': '/cdos/business/ucc/',
 'Business Entities in Colorado': '/cdos/business/business/',
 'Business Entity Transaction History': '/cdos/business/business/',
 'Trademarks for Businesses in Colorado': '/cdos/business/business/',
 'Trade Names for Businesses in Colorado': '/cdos/business/business/',
 'Master List in Colorado': '/cdos/business/business/',
 'Federal Tax-Exempt Subsection Codes in Colorado': '/cdos/business/nonprofit/',
 'Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit

In [71]:
dfDS = pd.read_csv("defs/xrefs.tsv",delimiter="\t")

In [72]:
dfDS.head()

,File,Title,Description
0,tax_exempt_cds.tsv,Federal Tax-Exempt Subsection Codes in Colorado,Tax exempt codes and descriptions
1,reg_finan.tsv,"Registration of Charities, Paid Solicitors, Pr...",Registration and financial data for Charitable...
2,offices.tsv,Charitable Organizations’ Offices in Colorado,Offices associated with Charitable Organizatio...
3,states.tsv,Other State Solicitation of Charities’ Registr...,States associated with Charitable Organization...
4,purpose.tsv,Charitable Purpose of the Charity in Colorado,Purpose of the Charitable Organization


In [73]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
   

    df = pd.DataFrame(tracker.get_all_records(head=3))

    return df
            
dfTracker = getXrefs()

In [74]:
dfTracker = dfTracker[['Dataset Title','Socrata Link']]

In [75]:
dfTracker.head()

,Dataset Title,Socrata Link
0,2019 Novel Coronavirus COVID-19 (2019-nCoV) Da...,rkuy-jxyz
1,Active Business Licenses Denver,s9wt-dsfz
2,Activities of Charities Operating in Colorado,vewn-5ajx
3,Airports in Colorado,x5bw-ax3d
4,All Special Districts in Colorado,dm2a-biqr


In [76]:
def mapInfo(row):
    title = row['Title'].strip()
    file = row['File'].strip()
    desc = row['Description']
    group=''
    if title in info:
        group = info[title]
        
    w4x4=""
    a = dfTracker.loc[dfTracker['Dataset Title'].str.strip() == title,'Socrata Link'].values.tolist()
    if len(a) > 0:
        w4x4=a[0]
        
    row['Tile'] = title
    row['File'] = file
    row['Description'] = desc
    row['Group'] = group
    row['4x4'] = w4x4
    
    return row

dfAll = dfDS.apply(mapInfo,axis=1)

In [77]:
display(dfAll[['Title','4x4']].head(20))

,Title,4x4
0,Federal Tax-Exempt Subsection Codes in Colorado,2z9k-uy4q
1,"Registration of Charities, Paid Solicitors, Pr...",37wu-kn3g
2,Charitable Organizations’ Offices in Colorado,3qtu-edua
3,Other State Solicitation of Charities’ Registr...,5wyf-xqw7
4,Charitable Purpose of the Charity in Colorado,7jm9-f28m
5,Paid Solicitor Solicitation Notices in Colorado,ew9y-6tv9
6,Campaign Reports for Solicitation Notices to C...,fdcw-ei67
7,Solicitation Campaign Supervisors Listed on So...,hyr8-d3v9
8,Charity Extension Requests,icqv-mi3c
9,Persons Associated with Charitable Organizatio...,mr4v-jz8u


In [78]:
dfTracker.loc[dfTracker['Dataset Title'].str.contains("Other Names"),'Dataset Title'].values

array(['Other Names a Registered Entity Uses to Solicit Contributions in Colorado'],
      dtype=object)

In [79]:
dfAll.head()

,File,Title,Description,Tile,Group,4x4
0,tax_exempt_cds.tsv,Federal Tax-Exempt Subsection Codes in Colorado,Tax exempt codes and descriptions,Federal Tax-Exempt Subsection Codes in Colorado,/cdos/business/nonprofit/,2z9k-uy4q
1,reg_finan.tsv,"Registration of Charities, Paid Solicitors, Pr...",Registration and financial data for Charitable...,"Registration of Charities, Paid Solicitors, Pr...",/cdos/business/nonprofit/,37wu-kn3g
2,offices.tsv,Charitable Organizations’ Offices in Colorado,Offices associated with Charitable Organizatio...,Charitable Organizations’ Offices in Colorado,/cdos/business/nonprofit/,3qtu-edua
3,states.tsv,Other State Solicitation of Charities’ Registr...,States associated with Charitable Organization...,Other State Solicitation of Charities’ Registr...,/cdos/business/nonprofit/,5wyf-xqw7
4,purpose.tsv,Charitable Purpose of the Charity in Colorado,Purpose of the Charitable Organization,Charitable Purpose of the Charity in Colorado,/cdos/business/nonprofit/,7jm9-f28m


In [95]:
for index,row in dfAll.iterrows():
    file=row['File']
    d4x4=row['4x4']
    fo=file[:-4]
#    file=
    df=pd.read_csv(f"defs/{file}",delimiter="\t")
    fileOut = f"{fo}_{d4x4}_src_fld_defs.json"
    print(fileOut)
    a = {row['Field']:row['Description'] for index,row in df.iterrows()}
    with open(f"defs/{fileOut}", "w") as outfile: 
        json.dump(a, outfile)
    outfile.close()
    

tax_exempt_cds_2z9k-uy4q_src_fld_defs.json
reg_finan_37wu-kn3g_src_fld_defs.json
offices_3qtu-edua_src_fld_defs.json
states_5wyf-xqw7_src_fld_defs.json
purpose_7jm9-f28m_src_fld_defs.json
sol_ntcs_ew9y-6tv9_src_fld_defs.json
sn_cmpgn_rpts_fdcw-ei67_src_fld_defs.json
persons_sols_ntcs_hyr8-d3v9_src_fld_defs.json
char_orgs_ext_icqv-mi3c_src_fld_defs.json
persons_entity_mr4v-jz8u_src_fld_defs.json
doing_bus_as_q2av-rpr5_src_fld_defs.json
char_orgs_sol_wwbh-7bpa_src_fld_defs.json
sol_ntcs_locs_wwhd-vg25_src_fld_defs.json
sol_typ_sol_ntcs_34aw-ny67_src_fld_defs.json
sol_typ_entity_w6kb-3vsj_src_fld_defs.json


In [84]:
df.to_dict(orient='record')

/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/pandas/core/frame.py:1490: FutureWarning: Using short name for 'orient' is deprecated. Only the options: ('dict', list, 'series', 'split', 'records', 'index') will be used in a future version. Use one of the above to silence this warning.
  FutureWarning,


[{'Field': 'Entity Id', 'Description': 'Entity ID '},
 {'Field': 'Document Id', 'Description': 'Document ID '},
 {'Field': 'Ce Fein',
  'Description': 'Federal Employer Identification Number '},
 {'Field': 'Org Name',
  'Description': 'Charitable Organization, Paid Solicitor or Professional Fundraising Consultant name '},
 {'Field': 'Sol Type Dscrp',
  'Description': 'Type of communication; i.e. door to door, direct mail, telephone, etc. '}]

In [87]:
df.groupby('Field')['Description'].apply().to_dict()

TypeError: apply() missing 1 required positional argument: 'func'